**Imports**

In [252]:
import numpy as np
import pandas as pd
import os
import csv
from scipy.optimize import Bounds # Bounds for Optimization
from scipy.optimize import LinearConstraint # Linear Constraints for Optimization
from scipy.optimize import milp # MILP Optimization Method
import tkinter as tk
from tkinter import ttk
import TKinterModernThemes as TKMT
from tkinter import *

In [253]:
out = []
pick1 = []
pick2 = []
pick3 = []
pick4 = []
names = []
x = []

**Optimization Method (MILP)**

In [254]:


def opt():
    global pick1,pick2,pick3,pick4,names,x
    input_file_1 = os.path.join("input_student", student_file_dropdown.get()) # type: ignore
    input_file_2 = os.path.join("input_project", project_file_dropdown.get()) # type: ignore
    student_data = pd.read_csv(input_file_1)
    project_data = pd.read_csv(input_file_2)

    project_names = project_data["project_name"]
    names = student_data["name"]

    # Capstone selections for individual students.
    pick1 = student_data["pick_1"]
    pick2 = student_data["pick_2"]
    pick3 = student_data["pick_3"]
    pick4 = student_data["pick_4"]
    # Get student names, project names, and student rankings for optimization. 
    num_students = len(names)
    num_projects = len(project_names)
    rankings = student_data[["pick_1", "pick_2", "pick_3", "pick_4"]].replace(0, np.nan).to_numpy() # Ignore 0s as they represent no selection.

    # Higher preference rankings get higher values (1 being highest value).
    value = np.zeros((num_students, num_projects))

    for i in range(num_students):
        for rank_pos in range(rankings.shape[1]):  # 4 positions/picks
            if np.isnan(rankings[i, rank_pos]):  # If the ranking is a NaN value, ignore it.
                continue
            project_index = int(rankings[i, rank_pos]) - 1
            if 0 <= project_index < num_projects:
                    value[i, project_index] = max(value[i, project_index], num_projects - rank_pos)

    # Check for duplicate rankings for each student and print a warning if one is found.
    for i, row in enumerate(rankings):
        valid = [p for p in row if not np.isnan(p)]
        if len(set(valid)) != len(valid):
             pass
            #print(f"Warning: Student {names[i]} has duplicate project rankings")

    # Maximum amounts of students in each capstone group 
    # (equal number of students in groups and remainder of students spread out to first few available projects).
    base = num_students // num_projects
    remainder = num_students % num_projects

    capacities = []
    for i in range(num_projects):
        if i < remainder: capacities.append(base + 1)
        else: capacities.append(base)

    capacities = np.array(capacities)

    # Objective Function: Negate values for maximizing student preferences per each project.
    c = -value.flatten()

    # List to store constraints. 
    constraints = []

    # Equality Constraint Matrix
    A_eq_students = np.zeros((num_students, num_students * num_projects))

    # Each row is a student and each student is constrained to be a part of one project.
    for i in range(num_students):
        for j in range(num_projects):
            A_eq_students[i, i * num_projects + j] = 1

    # Each student assigned to one project.
    b_eq_students = np.ones(num_students)

    # Add equality constraints to constraint list.
    constraints.append(LinearConstraint(A_eq_students, b_eq_students, b_eq_students))

    # Inequality Constraint Matrix
    A_ub_projects = np.zeros((num_projects, num_students * num_projects))

    # Each column is a project and counts how many students are attached to it.
    for j in range(num_projects):
        for i in range(num_students):
            A_ub_projects[j, i * num_projects + j] = 1

    # Add inequality constraints to constraint list.
    constraints.append(LinearConstraint(A_ub_projects, -np.inf, capacities))

    # Each variable is binary: 0 not in group, 1 is + integrality is binary
    bounds = Bounds(0, 1)
    integrality = np.ones(num_students * num_projects, dtype=int)

    # Run the MILP optimization method. Reshape the solution into a matrix.
    result = milp(c=c, constraints=constraints, integrality=integrality, bounds=bounds)

    if result.x is None:
            raise ValueError(f"Optimization failed: {result.message}")

    x = result.x.reshape((num_students, num_projects)).astype(int)

    for i in range(len(project_names)):
         lb.insert(i,project_names[i]) # type: ignore
    #preparing output list
    for i in range(len(project_names)):
        temp = []
        temp.append(project_names[i])
        for j in range(len(names)):
            if x[j, i] == 1:
                temp.append(names[j])
        out.append(temp)


**Display Names as They are Chosen/Assigned**

In [255]:
def on_select(event):
    print(names)
    first_pick = []
    second_pick = []
    assigned = []
    selected_indices = event.widget.curselection()
    if selected_indices:
        index = selected_indices[0]     #item index
        value = event.widget.get(index) #project name
        print(value)
        for i in range(len(names)):
            if pick1[i] == index+1:
                first_pick.append(names[i])
        label1.configure(text = f"Students that chose {value} as their first pick: {str(first_pick)}") # type: ignore

        for i in range(len(names)):
            if pick2[i] == index+1:
                second_pick.append(names[i])
        label2.configure(text = f"Students that chose {value} as their second pick: {str(second_pick)}") # type: ignore

        for i in range(len(names)):
            if x[i, index] == 1:
                assigned.append(names[i])
        label3.configure(text = f"Assigned to {value}: {str(assigned)}") # type: ignore

In [256]:
def button_clicked():
    save_path = "output"
    fileName = os.path.join(save_path, "out.csv")
    print(fileName)
    with open(fileName, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(out)

**UI Creation**

In [257]:
project_files = os.listdir("input_project")
student_files = os.listdir("input_student")



#app window initialization
root = tk.Tk()
root.config(background="white")
root.config(padx=10,pady=15)
root.title("Project Assigner")
screenheight = root.winfo_screenheight()
screenwidth = root.winfo_screenwidth()
alignscreen = '%dx%d' % (screenwidth/2,screenheight/2)
root.geometry(alignscreen)


#initializing UI items
title = tk.Label(
    root, text="Capstone Project Assigner", 
    font=("Arial",14),
    background="white")

student_file_dropdown = ttk.Combobox(root,values=student_files)
student_file_dropdown.set(student_files[0])
project_file_dropdown = ttk.Combobox(root,values=project_files)
project_file_dropdown.set(project_files[0])

opt_button = Button(root,text = 'Optimize', command = opt)
label1 = tk.Label(root,
                  background="white",
                  justify="center",
                  pady=3)
label2 = tk.Label(root,
                  background="white",
                  justify="center",
                  pady=3)
label3 = tk.Label(root,
                  background="white",
                  justify="center",
                  pady=3)
button = tk.Button(root, 
                   text="Output to csv", 
                   command=button_clicked,
                   activebackground="blue", 
                   activeforeground="white",
                   anchor="center",
                   bd=3,
                   bg="lightgray",
                   cursor="hand2",
                   disabledforeground="gray",
                   fg="black",
                   font=("Arial", 8),
                   height=1,
                   highlightbackground="black",
                   highlightcolor="green",
                   highlightthickness=1,
                   justify="center",
                   overrelief="raised",
                   padx=10,
                   pady=5,
                   width=10,
                   wraplength=100)

lb = tk.Listbox(root)
scrollbar = Scrollbar(lb)



title.place(relx=.5,y=0,anchor=CENTER)
student_file_dropdown.place(relx=0, rely=.07,relwidth=.2)
project_file_dropdown.place(relx=0,rely=.14,relwidth=.2)
opt_button.place(relx=0,rely=.21,relwidth=.2)
lb.bind('<<ListboxSelect>>', on_select)
lb.place(relx=.5,rely=.25,relwidth=.6,relheight=.37,anchor=CENTER)
lb.config(width=50,yscrollcommand=scrollbar.set)


scrollbar.pack(side = RIGHT, fill = BOTH)
scrollbar.config(command = lb.yview)

ttk.Separator(root).place(relx=0.5, rely=.45, relwidth=1,height=3,anchor=CENTER)

label1.place(x=0.5,rely=.46,relwidth=1,relheight=.1)
label2.place(x=0.5,rely=.56,relwidth=1,relheight=.1)
label3.place(x=0.5,rely=.66,relwidth=1,relheight=.1)

ttk.Separator(root).place(relx=0.5, rely=.76, relwidth=1,height=3,anchor=CENTER)

button.place(relx=.5,rely=.82,anchor=CENTER)
#start app
root.mainloop()


0      AA
1      AB
2      AC
3      AD
4      AE
       ..
494    TA
495    TB
496    TC
497    TD
498    TE
Name: name, Length: 499, dtype: str
project1
0      AA
1      AB
2      AC
3      AD
4      AE
       ..
494    TA
495    TB
496    TC
497    TD
498    TE
Name: name, Length: 499, dtype: str
project3
0      AA
1      AB
2      AC
3      AD
4      AE
       ..
494    TA
495    TB
496    TC
497    TD
498    TE
Name: name, Length: 499, dtype: str
project4
0      AA
1      AB
2      AC
3      AD
4      AE
       ..
494    TA
495    TB
496    TC
497    TD
498    TE
Name: name, Length: 499, dtype: str
project1
output\out.csv
